# 1. NaiveRAG 시스템 구현

- 전체 문서 파일을 바탕으로 NaiveRAG를 구현하고 Langfuse를 사용하여 추적해봅니다.

## 환경설정

In [51]:
import os

# 1. macOS OpenMP 충돌 방지 (반드시 최상단 실행)
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# 2. 토크나이저 병렬 처리 충돌 방지
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("환경 설정 완료. 이제 다음 셀들을 실행하세요.")

환경 설정 완료. 이제 다음 셀들을 실행하세요.


In [52]:
from dotenv import load_dotenv

load_dotenv()

True

## RAG 기본 파이프라인(skeleton code)

In [53]:
# 라이브러리 임포트

from langchain_text_splitters import RecursiveCharacterTextSplitter     # 분할기
from langchain_community.document_loaders import PyMuPDFLoader          # 로더
from langchain_community.vectorstores import FAISS                      # 벡터스토어
from langchain_core.output_parsers import StrOutputParser              
from langchain_core.runnables import RunnablePassthrough                
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langfuse.callback import CallbackHandler                           # llm 관측 ㄷ                                            # 경로


### 사전 준비단계

In [54]:
# 단계 1: 문서 로드
import os                       
import json
from pathlib import Path  

pdf_dir = '/Users/apple/Team2-RAG-Project/data/pdfs'
parsed_dir = '/Users/apple/Team2-RAG-Project/data/parsed'
os.makedirs(parsed_dir, exist_ok=True)

pdf_files = sorted([f for f in os.listdir(pdf_dir)])

for file_name in pdf_files:
    path = os.path.join(pdf_dir, file_name)
    loader = PyMuPDFLoader(path)
    docs = loader.load()

    out_path = os.path.join(parsed_dir, f"{os.path.splitext(file_name)[0]}.jsonl")
    with open(out_path, "w", encoding="utf-8") as f:
        for d in docs:
            record = {
                "source_file": file_name,          # 최소 메타데이터 1
                "page": d.metadata.get("page"),     # 최소 메타데이터 2
                "text": d.page_content
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict



- 대용량 파일 파싱, 후속 정제/디버깅에 용이하도록 data/parsed에 jsonl파일로 저장
- txt/jsonl/markdown/html 중. 
정보의 명확한 구분과 파일 자체에 내장된 메타데이터인 page 보존을 위해 jsonl파일을 선택

In [55]:
# 단계 1-2: 데이터 전처리

import json, re, unicodedata
from pathlib import Path
from collections import Counter

# [입력/출력 경로] data/parsed -> data/preprocessed
src_dir = Path("/Users/apple/Team2-RAG-Project/data/parsed")
dst_dir = Path("/Users/apple/Team2-RAG-Project/data/preprocessed")
dst_dir.mkdir(parents=True, exist_ok=True)

# 페이지 번호 헤더 패턴 예: "- 1 -"
page_no = re.compile(r"^-\s*\d+\s*-$")

# [입력] data/parsed의 각 jsonl 파일을 순회
for fp in src_dir.glob("*.jsonl"):
    rows = [json.loads(line) for line in fp.open("r", encoding="utf-8")]
    pages = [r.get("text", "") for r in rows]

    # [전처리-헤더/푸터 탐지] 페이지 상단/하단에서 반복되는 줄 찾기
    top, bottom = Counter(), Counter()
    for t in pages:
        ls = [x.strip() for x in t.splitlines() if x.strip()]
        top.update(ls[:2])       # 상단 2줄 후보
        bottom.update(ls[-2:])   # 하단 2줄 후보
    n = max(1, len(pages))
    repeated = {k for k, v in top.items() if v / n >= 0.5} | {k for k, v in bottom.items() if v / n >= 0.5}

    # [출력] 같은 파일명으로 data/preprocessed에 저장
    out_path = dst_dir / fp.name
    with out_path.open("w", encoding="utf-8") as out:
        for r in rows:
            s = r.get("text", "")

            # [전처리-제어문자 제거]
            s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C" or ch in "\n\t")

            # [전처리-헤더/푸터 제거] 반복줄 + 페이지번호 패턴 제거
            s = "\n".join(
                ln for ln in s.splitlines()
                if ln.strip() not in repeated and not page_no.match(ln.strip())
            )

            # [전처리-공백/줄바꿈 정리]
            s = re.sub(r"\r\n?", "\n", s)   # 줄바꿈 통일
            s = re.sub(r"[ \t]+", " ", s)   # 연속 공백/탭 -> 1칸
            s = re.sub(r"\n{2,}", "\n", s)  # 연속 빈 줄 축소
            s = s.strip()

            r["text"] = s
            out.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"완료: {dst_dir}")



완료: /Users/apple/Team2-RAG-Project/data/preprocessed


- 제어문자 제거: 보이지 않는 제어문자(\n, \t 제외) 삭제
- 헤더/푸터 제거: 페이지 상/하단에서 반복되는 줄 + - 1 - 같은 페이지 번호 줄 제거
- 공백/줄바꿈 정리: 연속 공백/탭 축소, 줄바꿈 형식 통일, 빈 줄 과다 제거

In [71]:
# 1-3) 메타데이터와 결합, Document 객체 생성
import re, unicodedata
import pandas as pd
from langchain_core.documents import Document

# 1) 경로
csv_path = "/Users/apple/Team2-RAG-Project/data/raw/data_list.csv"
pre_dir = "/Users/apple/Team2-RAG-Project/data/preprocessed"

# 2) 파일명 키(매칭용) 함수

def file_key(name):
    s = str(name).rsplit(".", 1)[0]
    s = unicodedata.normalize("NFC", s)  # 핵심
    s = re.sub(r"\s+", " ", s).strip()
    return s


# 3) CSV -> {파일키: 메타데이터} 딕셔너리
meta_df = pd.read_csv(csv_path)
meta_df["file_key"] = meta_df["파일명"].apply(file_key)
meta_map = meta_df.set_index("file_key").to_dict("index")

# 4) preprocessed jsonl + CSV 메타 결합 -> Document 생성
documents = []
for fname in sorted(os.listdir(pre_dir)):
    if not fname.endswith(".jsonl"):
        continue

    k = file_key(fname)               # jsonl 파일명 키
    base_meta = meta_map.get(k, {})   # csv 메타데이터

    with open(os.path.join(pre_dir, fname), "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            text = (row.get("text") or "").strip()
            if not text:
                continue

            md = dict(base_meta)
            md["source_file"] = row.get("source_file")
            md["page"] = row.get("page")
            md["author"] = row.get("author")

            documents.append(Document(page_content=text, metadata=md))

print("Document 개수:", len(documents))
print("샘플 metadata:", documents[0].metadata if documents else {})


Document 개수: 7547
샘플 metadata: {'공고 번호': '20240330003', '공고 차수': 0.0, '사업명': '2024년 벤처확인종합관리시스템 기능 고도화 용역사업 입찰공고', '사업 금액': 352000000.0, '발주 기관': '(사)벤처기업협회', '공개 일자': '2024-03-19 10:49:46', '입찰 참여 시작일': nan, '입찰 참여 마감일': '2024-04-09 15:00:00', '사업 요약': '- 2024년 벤처확인종합관리시스템 기능 고도화 용역사업\n- 복수의결권주식, 스톡옵션, 성과조건부주식교부 기능 구축 및 고도화\n- 벤처기업법에 따라 발행된 복수의결권주식 보고 업무 처리 시스템 구축\n- 벤처기업법에 따라 부여된 벤처기업 스톡옵션 부여, 취소-철회 신고 및 업무시스템 구축\n- 벤처기업법에 따라 성과조건부주식교부계약의 신고 및 업무처리 시스템 구축\n- 복수의결권주식 발행 보고 및 스톡옵션 신고기능은 기존 시스템 내 벤처확인 이력정보와 신고자(벤처기업)의 정보와 연동', '파일형식': 'hwp', '파일명': '(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp', '텍스트': '    \r\n \r\n2024년  벤처확인종합관리시스템 기능 고도화  용역사업\r\n(복수의결권주식, 스톡옵션, 성과조건부주식) -\r\n제안요청서\r\n2024. 03.   \r\n \r\n \r\n   \r\n \r\n목  차\r\n 1. 추진개요\t  -   \t 3\r\n   \r\n 2. 추진방안\t  -   \t 5\r\n   \r\n 3. 추진내용\t  -   \t 9\r\n   \r\n 4. 제안요청내용\t  -   \t 24\r\n   \r\n 5. 입찰관련사항\t  -   \t 78\r\n   \r\n 6. 제안서작성요령\t  -   \t 82\r\n 7. 별지서식 및 붙임\t  -   \t 94\r\n \r\nⅠ. 추진개요\r\n \r\n1\r\n추진배경 및 방향\r\

- 전처리 단계에서 공백이었던 29페이지가 삭제되어 (7576 - 29) 총 7547개의 Document 객체가 생성되었다.
- data_list.csv 메타데이터와 결합되었다.

In [57]:
# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(documents)

In [58]:
len(split_documents)

10915

- 총 토큰 수는 7184419개이며,  
 chunk_size=1000에 chunk_overlap=50으로. 
 총 10915개의 chunk가 만들어졌다.

In [59]:
print(split_documents[1].page_content)

목 차
 1. 추진개요·························································································· 3
 2. 추진방안·························································································· 5
 3. 추진내용·························································································· 9
 4. 제안요청내용················································································ 24
 5. 입찰관련사항················································································ 78
 6. 제안서작성요령············································································ 82
 7. 별지서식 및 붙임········································································ 94


In [60]:
# 단계 3: 임베딩(Embedding) 생성
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small", chunk_size=100)
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x17cd3e650>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x17cd3e800>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version='', openai_api_base=None, openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=100, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [72]:
# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

In [73]:
# 단계 5: 검색기(Retreiever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x3340efdc0>)

In [74]:
retriever.invoke("국립중앙의료원의 응급환자 전원 지원 시스템의 핵심 프로세스 중 '상황의사'의 주된 역할은 무엇인가요?")

[Document(id='e09b5811-6d9c-4330-9ebc-36d1b2efa358', metadata={'source_file': '국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.pdf', 'page': 7, 'author': None}, page_content='1\n 목표시스템 구성도\n - 지리적으로 본관(국립중앙의료원), 삼화빌딩(중앙상황실, 광역상황실(서울)), \n광역상황실(대전, 대구, 광주)로 구분할 수 있음\n - 물리적으로 같은 장소에 위치한 중앙상황실과 수도권 광역상황실(서울)은 \n같은 장비를 사용\n - 안정적인 서비스 제공 및 장애 대비를 위한 이중화 구성\n - 주 통신장비는 삼화빌딩 위치. 이와 통신을 위한 개별장비는 각 지역의 \n광역상황실에 구축\n※ 최종목표시스템구성도는발주처, 제안사간의상호논의후, 최적구성도를도출및\n제안하여야함\n2\n 업무 흐름도\n 가. 상황접수(유선) 흐름도\n 나. 업무절차\n - 119 이송 중증응급환자에 대한 전원 지원\n절차\n담당\n세부내용\n접수\n상황요원\n• 소방청중앙119구급상황관리센터로부터전화요청\n• 사전의뢰기관, 추가정보(협진망등) 확인\n▼\n환자정보\n수집\n상황의사\n• 접수시스템에환자정보입력\n• 필요시구급대원과통화하여환자정보수집(중앙119구급\n상황관리센터와협의필요)\n▼\n의뢰\n상황의사\n• 사전의뢰기관, 지역별이송지침등을확인하여의뢰대상\n기관선정\n• 수용의뢰진행\n상황요원\n• 통화건별녹취이력, 데이터입력\n• 응답대기중인기관에서5분이상답변지연시진행상황확인\n▼\n(수용기관\n결정)\n(상황의사)\n• (수용가능병원선정불가시, 지역별이송지침에따라환자\n수용기관을결정)\n• (수용기관에환자이송함을통보)\n▼\n수용기관\n통보\n상황의사\n• 구급상황관리센터로통보\n▼\n이송\n확인\n상황요원\n• 수용병원에환자수용및진료여부확인'),
 Document(id='d

In [75]:
# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks.
Use the following poeces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Answer in Korean.

#Question:
{question}
#Context:
{context}

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 를 생성합니다.
llm = ChatOpenAI(model_name="gpt-5-mini", temperature=1)

# 단계 8: 체인(Chain) 생성
langfuse_handler = CallbackHandler() 
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [76]:
# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "국립중앙의료원의 응급환자 전원 지원 시스템의 핵심 프로세스 중 '상황의사'의 주된 역할은 무엇인가요?"
responce = chain.invoke(question,
                        config={"callbacks": [langfuse_handler]})
print(responce)

상황의사의 주된 역할은 응급전원 접수 처리와 병원 선정·조정입니다. 구체적으로는:
- 접수시스템에 환자정보를 입력하고(필요시 구급대원과 통화해 추가 임상정보 수집)
- 사전의뢰기관과 지역별 이송지침을 확인해 적절한 수용(전원) 대상 기관을 선정
- 수용 요청을 진행하고(수용기관 선정이 불가할 경우 지침에 따라 결정)
- 수용기관 및 중앙구급상황관리센터에 전원 계획을 통보·조정하는 업무를 담당합니다.


In [77]:
retrieved_chunks = retriever.invoke("국립중앙의료원의 응급환자 전원 지원 시스템의 핵심 프로세스 중 '상황의사'의 주된 역할은 무엇인가요?")
for i, doc in enumerate(retrieved_chunks):
    page_num = doc.metadata.get("page", "정보 없음")
    print(page_num) 

7
1
3
2


In [78]:
question = "나노종합기술원의 스마트 팹 설비온라인 시스템 구축 용역은 계약 후 기간이 총 얼마나 소요되나요?"
responce = chain.invoke(question,
                        config={"callbacks": [langfuse_handler]})
print(responce)

계약체결일로부터 18개월(약 1년 6개월)입니다.


In [79]:
retrieved_chunks = retriever.invoke("나노종합기술원의 스마트 팹 설비온라인 시스템 구축 용역은 계약 후 기간이 총 얼마나 소요되나요?")
for i, doc in enumerate(retrieved_chunks):
    page_num = doc.metadata.get("page", "정보 없음")
    print(page_num) 

0
2
27
25


#### 평가 데이터셋
- Q1. "국립중앙의료원의 응급환자 전원 지원 시스템의 핵심 프로세스 중 '상황의사'의 주된 역할은 무엇인가요?"

- A."의뢰 대상 기관 선정, 수용 의뢰 및 상황 지원을 담당합니다."

- 14페이지 [업무 흐름도] 설명부, 이미지 정보

- Q2. "나노종합기술원의 스마트 팹 설비온라인 시스템 구축 용역은 계약 후 기간이 총 얼마나 소요되나요?"

- A."총 18개월입니다."

- "1페이지 15째 줄"

#### 결과:
- Q1: '수용의뢰'는 맞췄으나, 상황 지원은 맞추지 못함
- Q2: 18개월을 정확히 맞추었다.

In [80]:
# 평가

def run_evaluation(chain, threshold=0.35, use_langfuse=False):
    dataset_path = "/Users/apple/Team2-RAG-Project/evaluation/test_questions.json"
    with open(dataset_path, "r", encoding="utf-8") as f:
        dataset = json.load(f)

    results = []
    correct_count = 0
    print("RAG 시스템 평가 시작...\n")

    for i, item in enumerate(dataset, 1):
        q = item["question"]
        gold = item["answer"]

        try:
            prediction = chain.invoke(q)
        except Exception as e:
            prediction = f"[ERROR] {e}"

        if not isinstance(prediction, str):
            prediction = str(prediction)

        score = _soft_score(prediction, gold)
        ok = score >= threshold
        if ok:
            correct_count += 1

        results.append({
            "id": item.get("id", i),
            "question": q,
            "gold": gold,
            "prediction": prediction,
            "score": score,
            "correct": ok,
        })

        print(f"Q{i:02d}: [{'O' if ok else 'X'}] score={score:.3f} | 질문: {q[:30]}...")

    print(f"\n평가 완료: {correct_count} / {len(results)}")
    return results


In [82]:
results = run_evaluation(chain, threshold=0.35, use_langfuse=False)

RAG 시스템 평가 시작...

Q01: [O] score=0.495 | 질문: 봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총...
Q02: [O] score=0.588 | 질문: 이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무...
Q03: [X] score=0.233 | 질문: 봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 ...
Q04: [O] score=0.452 | 질문: 고려대 차세대 학사시스템 구축 사업의 전체 예산 규모와...
Q05: [X] score=0.097 | 질문: 고려대 포털 고도화 작업 중에 기존 'AI선배' 서비스...
Q06: [X] score=0.185 | 질문: 고려대 학사행정시스템을 구축할 때 서울과 세종 캠퍼스별...
Q07: [O] score=0.917 | 질문: 중앙의료원 응급의료 상황관리시스템 구축 사업에서 광역상...
Q08: [X] score=0.063 | 질문: 국립중앙의료원 인프라 요건 중 DB 서버(Server-...
Q09: [X] score=0.155 | 질문: 응급환자 전원 지원 시스템의 핵심 업무 중 '상황의사'...
Q10: [O] score=0.467 | 질문: 나노종합기술원의 스마트 팹 설비온라인 시스템 구축 용역...
Q11: [X] score=0.071 | 질문: 기술원의 'LoT Tracking' 환경 구축을 위해서...
Q12: [X] score=0.093 | 질문: 나노종합기술원 보안 요건 중에서 DB 암호화 제품이 반...
Q13: [O] score=0.500 | 질문: 아시아물위원회에서 추진하는 우즈벡-키르기스스탄 기후대응...
Q14: [X] score=0.074 | 질문: 우즈벡 스마트 관개사업 전문가 파견 시, '수자원 전문...
Q15: [X] score=0.042 | 질문: 우즈벡-키르기스스탄 현지 지원용 기자재 목록에 포함된 ...
Q16: [X] score=0.063 | 질문: 어촌어항공단 차세대 경영관리시스템 사업에서 그룹웨어(G...
Q17: [

| 유형 | 문항 번호 (ID) | 문항 수 | 주요 평가 요소 |
|---|---|---:|---|
| 텍스트 | 5, 6, 16, 19, 22, 25, 26, 27, 28 | 9 | 문맥 파악 및 비정형 데이터 추출 |
| 표 | 1, 3, 4, 8, 10, 12, 14, 17, 20, 24, 29, 30 | 12 | 표 구조 분석 및 정확한 수치 매칭 |
| 이미지 | 2, 7, 9, 11, 13, 15, 18, 21, 23 | 9 | 시각 도식 해석 및 논리적 흐름 파악 |

정답: 1, 2, 4, 7, 10, 13
- 텍스트보다 이미지, 표에 있는 정보가 잘 추출된 것을 확인할 수 있다.
텍스트 형식으로 추출된 표에 정보가 잘 저장되있다거나, 이미지가 추출되지 않더라도 텍스트 형식으로도 문서에 있는 것으로 추정할 수 있다.